# 01 - Data Cleaning: Walmart Weekly Sales

This notebook performs the initial data cleaning and validation for the Walmart weekly sales dataset.

**Main steps:**
- Set up paths and import libraries.
- Load the raw `Walmart.csv` file.
- Inspect structure, data types, and basic issues.
- Standardise column names and data types.
- Check for missing values, duplicates, and basic sanity issues.
- Save a cleaned version of the dataset for further analysis.

In [3]:
# Import core libraries and set up file paths

import pandas as pd
import numpy as np
from pathlib import Path

# If notebook is inside /notebooks folder, root is parent directory
PROJECT_ROOT = Path.cwd().resolve().parent

# Define data folders
DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_CLEAN = PROJECT_ROOT / "data" / "clean"

# Define raw file path
RAW_FILE_NAME = "Walmart.csv"
RAW_FILE_PATH = DATA_RAW / RAW_FILE_NAME

# Ensure clean data folder exists
DATA_CLEAN.mkdir(parents=True, exist_ok=True)

print("Notebook working directory:", Path.cwd())
print("Project root:", PROJECT_ROOT)
print("Raw file path:", RAW_FILE_PATH)
print("Exists?", RAW_FILE_PATH.exists())


Notebook working directory: C:\Users\Mist\Documents\Portfolio\P1. Retail Sales Forecasting and Stock Optimisation\notebooks
Project root: C:\Users\Mist\Documents\Portfolio\P1. Retail Sales Forecasting and Stock Optimisation
Raw file path: C:\Users\Mist\Documents\Portfolio\P1. Retail Sales Forecasting and Stock Optimisation\data\raw\Walmart.csv
Exists? True


## Load raw Walmart sales data

In this step we:

- Load the original `Walmart.csv` file from the `data/raw` folder.
- Quickly inspect the shape and the first few rows to confirm it has loaded correctly.


In [4]:
# Load the raw Walmart.csv data and inspect basic structure

# Read the CSV file
df_raw = pd.read_csv(RAW_FILE_PATH)

# Show basic shape and first few rows
print("Raw data shape:", df_raw.shape)
df_raw.head()


Raw data shape: (6435, 8)


,Store,Date,Weekly_Sales,Holiday_Flag,Temperature,Fuel_Price,CPI,Unemployment
0,1,05-02-2010,1643690.90,0,42.31,2.572,211.096358,8.106
1,1,12-02-2010,1641957.44,1,38.51,2.548,211.242170,8.106
2,1,19-02-2010,1611968.17,0,39.93,2.514,211.289143,8.106
3,1,26-02-2010,1409727.59,0,46.63,2.561,211.319643,8.106
4,1,05-03-2010,1554806.68,0,46.50,2.625,211.350143,8.106


## Inspect data types and non-null counts

Here we:

- Use `.info()` to inspect column data types.
- Check non null counts to see if there are missing values.
- Confirm whether the `Date` column is still a string and needs conversion to datetime.


In [5]:
# Inspect data types and non-null counts for each column

df_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6435 entries, 0 to 6434
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Store         6435 non-null   int64  
 1   Date          6435 non-null   object 
 2   Weekly_Sales  6435 non-null   float64
 3   Holiday_Flag  6435 non-null   int64  
 4   Temperature   6435 non-null   float64
 5   Fuel_Price    6435 non-null   float64
 6   CPI           6435 non-null   float64
 7   Unemployment  6435 non-null   float64
dtypes: float64(5), int64(2), object(1)
memory usage: 402.3+ KB


## Create a working copy of the raw data

Here we:

- Create a copy of the raw DataFrame (`df_raw`) into `df`.
- This keeps the original data untouched and avoids accidental modification.


In [6]:
# Create a working copy so we do not alter the original raw DataFrame
df = df_raw.copy()

print("Working DataFrame shape:", df.shape)
df.head()


Working DataFrame shape: (6435, 8)


,Store,Date,Weekly_Sales,Holiday_Flag,Temperature,Fuel_Price,CPI,Unemployment
0,1,05-02-2010,1643690.90,0,42.31,2.572,211.096358,8.106
1,1,12-02-2010,1641957.44,1,38.51,2.548,211.242170,8.106
2,1,19-02-2010,1611968.17,0,39.93,2.514,211.289143,8.106
3,1,26-02-2010,1409727.59,0,46.63,2.561,211.319643,8.106
4,1,05-03-2010,1554806.68,0,46.50,2.625,211.350143,8.106


## Standardise column names to snake_case

To keep things tidy and consistent in the project, we:

- Convert column names from mixed case (e.g. `Weekly_Sales`)
- To snake_case (e.g. `weekly_sales`).

This is a common convention and makes it easier to refer to columns in code.


In [7]:
# Rename columns from CamelCase to snake_case for consistency

rename_map = {
    "Store": "store",
    "Date": "date",
    "Weekly_Sales": "weekly_sales",
    "Holiday_Flag": "holiday_flag",
    "Temperature": "temperature",
    "Fuel_Price": "fuel_price",
    "CPI": "cpi",
    "Unemployment": "unemployment",
}

df = df.rename(columns=rename_map)

print("Columns after renaming:")
print(df.columns.tolist())
df.head()


Columns after renaming:
['store', 'date', 'weekly_sales', 'holiday_flag', 'temperature', 'fuel_price', 'cpi', 'unemployment']


,store,date,weekly_sales,holiday_flag,temperature,fuel_price,cpi,unemployment
0,1,05-02-2010,1643690.90,0,42.31,2.572,211.096358,8.106
1,1,12-02-2010,1641957.44,1,38.51,2.548,211.242170,8.106
2,1,19-02-2010,1611968.17,0,39.93,2.514,211.289143,8.106
3,1,26-02-2010,1409727.59,0,46.63,2.561,211.319643,8.106
4,1,05-03-2010,1554806.68,0,46.50,2.625,211.350143,8.106


## Convert `date` column to datetime and confirm data types

At the moment, `date` is stored as a text (`object`) field.

In this step we:

- Convert `date` to a proper `datetime` type using the known format `day-month-year`.
- Leave numeric columns as they already have correct numeric dtypes.
- Recheck `.info()` to confirm the structure.


In [8]:
# Convert 'date' column from string to datetime format (DD-MM-YYYY)

df["date"] = pd.to_datetime(df["date"], format="%d-%m-%Y", errors="coerce")

# Check final data types
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6435 entries, 0 to 6434
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   store         6435 non-null   int64         
 1   date          6435 non-null   datetime64[ns]
 2   weekly_sales  6435 non-null   float64       
 3   holiday_flag  6435 non-null   int64         
 4   temperature   6435 non-null   float64       
 5   fuel_price    6435 non-null   float64       
 6   cpi           6435 non-null   float64       
 7   unemployment  6435 non-null   float64       
dtypes: datetime64[ns](1), float64(5), int64(2)
memory usage: 402.3 KB


## Check for missing values

Here we inspect:

- How many missing values exist in each column.
- Whether any critical fields (`date`, `store`, `weekly_sales`) contain missing data.

If critical fields contain missing values, those rows must be removed.
If non-critical fields have missing values, we can decide later whether to impute or drop them.


In [9]:
# Check missing values across all columns

missing_counts = df.isna().sum().sort_values(ascending=False)
missing_counts


store           0
date            0
weekly_sales    0
holiday_flag    0
temperature     0
fuel_price      0
cpi             0
unemployment    0
dtype: int64

No missing values found, excellent!

## Check for duplicates

In this step we:

- Look for fully duplicated rows.
- Evaluate whether `(store, date)` combinations are duplicated.
- Decide whether to remove duplicates if any exist.

The Walmart dataset usually contains no duplicates, but we check as part of a professional workflow.


In [11]:
# Check fully duplicated rows
full_dupes = df.duplicated().sum()
print("Number of fully duplicated rows:", full_dupes)

# Check duplicates by (store, date)
if {"store", "date"} <= set(df.columns):
    dup_by_key = df.duplicated(subset=["store", "date"]).sum()
    print("Number of duplicated (store, date) pairs:", dup_by_key)


Number of fully duplicated rows: 0
Number of duplicated (store, date) pairs: 0


No duplicates found!

## Perform basic sanity checks

We check the following:

- `weekly_sales` should never be negative.
- `store` and `holiday_flag` should be valid integers.
- Values such as temperature, fuel price, CPI, and unemployment should be reasonable.

If negative weekly sales exist, we remove those rows.


In [14]:
# Check negative weekly sales
neg_sales_count = (df["weekly_sales"] < 0).sum()
print("Negative weekly sales rows:", neg_sales_count)

# If negative sales exist, remove them
if neg_sales_count > 0:
    before_rows = df.shape[0]
    df = df[df["weekly_sales"] >= 0]
    after_rows = df.shape[0]
    print(f"Removed {before_rows - after_rows} negative sales rows")

# Quick summary stats for numeric features
df.describe()

Negative weekly sales rows: 0


,store,date,weekly_sales,holiday_flag,temperature,fuel_price,cpi,unemployment
count,6435.000000,6435,6.435000e+03,6435.000000,6435.000000,6435.000000,6435.000000,6435.000000
mean,23.000000,2011-06-17 00:00:00,1.046965e+06,0.069930,60.663782,3.358607,171.578394,7.999151
min,1.000000,2010-02-05 00:00:00,2.099862e+05,0.000000,-2.060000,2.472000,126.064000,3.879000
25%,12.000000,2010-10-08 00:00:00,5.533501e+05,0.000000,47.460000,2.933000,131.735000,6.891000
50%,23.000000,2011-06-17 00:00:00,9.607460e+05,0.000000,62.670000,3.445000,182.616521,7.874000
75%,34.000000,2012-02-24 00:00:00,1.420159e+06,0.000000,74.940000,3.735000,212.743293,8.622000
max,45.000000,2012-10-26 00:00:00,3.818686e+06,1.000000,100.140000,4.468000,227.232807,14.313000
std,12.988182,NaN,5.643666e+05,0.255049,18.444933,0.459020,39.356712,1.875885


## Sort data and reset index

To prepare for time based analysis later:

- Sort the data by `store` and `date`.
- Reset the row index to keep the structure tidy.


In [15]:
# Sort the data for time based analysis

df = df.sort_values(["store", "date"]).reset_index(drop=True)

df.head()


,store,date,weekly_sales,holiday_flag,temperature,fuel_price,cpi,unemployment
0,1,2010-02-05,1643690.90,0,42.31,2.572,211.096358,8.106
1,1,2010-02-12,1641957.44,1,38.51,2.548,211.242170,8.106
2,1,2010-02-19,1611968.17,0,39.93,2.514,211.289143,8.106
3,1,2010-02-26,1409727.59,0,46.63,2.561,211.319643,8.106
4,1,2010-03-05,1554806.68,0,46.50,2.625,211.350143,8.106


## Save cleaned dataset

We now save the cleaned dataset in the `data/clean/` folder as:

`walmart_sales_clean.csv`

This file will be used by the following EDA and modelling notebooks.


In [16]:
# Save the cleaned dataset

clean_path = DATA_CLEAN / "walmart_sales_clean.csv"
df.to_csv(clean_path, index=False)

print("Clean dataset saved to:", clean_path)
print("Final shape:", df.shape)


Clean dataset saved to: C:\Users\Mist\Documents\Portfolio\P1. Retail Sales Forecasting and Stock Optimisation\data\clean\walmart_sales_clean.csv
Final shape: (6435, 8)
